# AutoLab — téléchargement unique du checkpoint UMA (uma-s-1p2) sur Kaggle

À lancer **une seule fois depuis l'interface Kaggle**, sans GPU (Accelerator : None), avec Internet activé et le secret `HF_TOKEN` attaché.
1. Télécharge depuis Hugging Face (`facebook/UMA`) les trois fichiers dont fairchem a besoin : le checkpoint `uma-s-1p2.pt` et deux fichiers d'énergies de référence.
2. Les range dans `/kaggle/working/fairchem_cache/`, au format attendu par fairchem (`FAIRCHEM_CACHE_DIR`), avec un `manifest.json` (tailles et empreintes SHA-256).
3. Vérifie que fairchem charge le modèle **hors ligne, sans jeton** (énergie de LiCl sur CPU).

Ensuite, rien d'autre à faire : les notebooks de calcul envoyés par l'API attachent la sortie de ce notebook comme source (`kernel_sources`) et chargent UMA sans aucun secret. Le jeton n'est jamais affiché. Ne pas renvoyer ce notebook par l'API : le secret serait détaché.

In [ ]:
# Download the three files fairchem needs for uma-s-1p2 (same arguments as fairchem.core.calculate.pretrained_mlip),
# then copy them with symlinks resolved into the FAIRCHEM_CACHE_DIR layout. HF_TOKEN is never printed.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path

from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

FILES = [("checkpoints", "uma-s-1p2.pt"), ("references", "iso_atom_elem_refs.yaml"),
         ("references", "form_elem_refs.yaml")]
TMP = Path("/tmp/fairchem_download")
OUT = Path("/kaggle/working/fairchem_cache")

t0 = time.time()
token = UserSecretsClient().get_secret("HF_TOKEN")
for sub, name in FILES:
    hf_hub_download(repo_id="facebook/UMA", filename=name, subfolder=sub, cache_dir=str(TMP), token=token)
token = None
shutil.rmtree(OUT, ignore_errors=True)
shutil.copytree(TMP, OUT, symlinks=False, ignore=shutil.ignore_patterns("blobs", ".locks"))
shutil.rmtree(TMP)

manifest = {}
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for block in iter(lambda: f.read(1 << 20), b""):
                h.update(block)
        manifest[str(p.relative_to(OUT))] = {"bytes": p.stat().st_size, "sha256": h.hexdigest()}
Path("/kaggle/working/manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")
print(json.dumps(manifest, indent=1))
print(f"Téléchargement : {time.time() - t0:.0f} s, {sum(v['bytes'] for v in manifest.values()) / 1e9:.2f} Go")

In [ ]:
# Offline check on CPU, without any token: a copy of the folder (as the API notebooks will do) is enough for fairchem.
t0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fairchem-core==2.24.0"], check=True)
CHECK_DIR = Path("/tmp/fairchem_cache_check")
shutil.rmtree(CHECK_DIR, ignore_errors=True)
shutil.copytree(OUT, CHECK_DIR)
check = """
from ase.build import bulk
from fairchem.core import FAIRChemCalculator, pretrained_mlip
unit = pretrained_mlip.get_predict_unit("uma-s-1p2", device="cpu")
atoms = bulk("LiCl", "rocksalt", a=5.13)
atoms.calc = FAIRChemCalculator(unit, task_name="omat")
print("LiCl energy (CPU, offline):", round(float(atoms.get_potential_energy()), 4), "eV")
"""
env = {k: v for k, v in os.environ.items() if k not in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")}
env.update(FAIRCHEM_CACHE_DIR=str(CHECK_DIR), HF_HUB_OFFLINE="1")
r = subprocess.run([sys.executable, "-c", check], env=env, capture_output=True, text=True)
print(r.stdout[-1500:])
if r.returncode:
    print(r.stderr[-3000:])
shutil.rmtree(CHECK_DIR, ignore_errors=True)
status = "OK" if r.returncode == 0 else f"ÉCHEC (code {r.returncode})"
Path("/kaggle/working/offline_check.txt").write_text(status + "\n" + r.stdout[-1500:], encoding="utf-8")
print(f"Chargement hors ligne : {status} ({time.time() - t0:.0f} s)")